# BPE : Byte Pair Encoding

- BPE가 왜 필요한가?
	  언어모델은 텍스트를 token 단위로 쪼개서 처리하는데, 여기서 딜레마가 생긴다  
	  - 단어 단위 : 사전에 없는 단어는 전부 <UNK> 처리 -> 정보 손실  
	  - 글자 단위 : vocab은 작아지지만 시퀀스가 너무 길어짐 -> 문맥 학습 어려움  
	Byte Pair Encoding은 이 둘 사이의 절충안 : 자주 나오는 글자 조합을 통째로 하나의 토큰으로 묶어버리는 subword 방식  

- 방법 : 제일 자주 붙어 나오는 쌍을 계속 합치기   
	  1. 모든 단어를 글자 단위로 쪼개기  
	  2. 인접한 글자 쌍(pair) 중 가장 자주 등장하는 쌍 찾기  
	  3. 그 쌍을 하나의 새 토큰으로 합치기  
	  4. 원하는 vocab 크기가 될 때까지 2~3을 반복  


In [1]:
import re, collections

In [4]:
def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbol = word.split()
        for i in range(len(symbol) - 1):
            pairs[(symbol[i], symbol[i+1])] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(" ".join(pair))
    p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
    for word in v_in:
        w_out = p.sub("".join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

In [7]:
vocab = { 
    "l o w </w>": 5,
    "l o w e r </w>": 2,
    "n e w e s t </w>": 6,
    "w i d e s t </w>": 3,
}

num_merges = 10

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        print("더 이상 merge할 pair가 없어요!")
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"[Step {i+1}] Merged: {best}")
    print(f"Vocab: {vocab}\n")

[Step 1] Merged: ('e', 's')
Vocab: {'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w es t </w>': 6, 'w i d es t </w>': 3}

[Step 2] Merged: ('es', 't')
Vocab: {'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w est </w>': 6, 'w i d est </w>': 3}

[Step 3] Merged: ('est', '</w>')
Vocab: {'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w est</w>': 6, 'w i d est</w>': 3}

[Step 4] Merged: ('l', 'o')
Vocab: {'lo w </w>': 5, 'lo w e r </w>': 2, 'n e w est</w>': 6, 'w i d est</w>': 3}

[Step 5] Merged: ('lo', 'w')
Vocab: {'low </w>': 5, 'low e r </w>': 2, 'n e w est</w>': 6, 'w i d est</w>': 3}

[Step 6] Merged: ('n', 'e')
Vocab: {'low </w>': 5, 'low e r </w>': 2, 'ne w est</w>': 6, 'w i d est</w>': 3}

[Step 7] Merged: ('ne', 'w')
Vocab: {'low </w>': 5, 'low e r </w>': 2, 'new est</w>': 6, 'w i d est</w>': 3}

[Step 8] Merged: ('new', 'est</w>')
Vocab: {'low </w>': 5, 'low e r </w>': 2, 'newest</w>': 6, 'w i d est</w>': 3}

[Step 9] Merged: ('low', '</w>')
Vocab: {'low</w>': 5, 'low e r </w>': 2, 'newest<

In [8]:
print("=== 최종 Vocab ===")
for word, freq in vocab.items():
    print(f"{word}: {freq}")

=== 최종 Vocab ===
low</w>: 5
low e r </w>: 2
newest</w>: 6
wi d est</w>: 3


In [9]:
def get_vocab_tokens(vocab):
    tokens = set()
    for word in vocab:
        tokens.update(word.split())
    return tokens

print("최종 토큰 집합:", get_vocab_tokens(vocab))
print("토큰 개수:", len(get_vocab_tokens(vocab)))

최종 토큰 집합: {'e', 'd', 'r', 'est</w>', '</w>', 'low</w>', 'low', 'wi', 'newest</w>'}
토큰 개수: 9
